# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.utils.jupyter import display_shap_colors_df
from mllm_shap.shap import Explainer, ComplementaryShapExplainer

Define LiquidAudio model (this call loads it up to the memory!).

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history

W1111 19:57:38.575000 75049 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create compact explainer that will make initial call and then explain it using shapley values using Complementary Formula.

In [6]:
explainer = Explainer(
    model=model, shap_explainer=ComplementaryShapExplainer(fraction=0.015)
)

Create new chat instance and assign it messages.

In [7]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.NONE,  # calculate shapley values for all roles
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.USER)
chat.add_text("Who are you?")
chat.end_turn()

# Usage

Let's calculate shapley values for current conversation.

Generation kwargs allows to customize model interference - here we limit it to 4 tokens and change text_temperature from default 0.0 to 0.2, text_top_k from default 1 to 3. 

In [8]:
generation_kwargs = {
    "max_new_tokens": 4,
    "model_config": ModelConfig(text_temperature=0.2, text_top_k=3),
}

result = explainer(
    chat=chat,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2025-11-11 19:57:43,022 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-11-11 19:57:43,622 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 10 (up to 1023 additional calls)


Calculating SHAP values:   0%|          | 0/14 [00:00<?, ?it/s]

Let's see final Shap values.

In [9]:
display_shap_colors_df(
    pd.DataFrame(
        list(
            zip(
                [chat.decode_text(token) for token in result.full_chat.input_tokens],
                result.full_chat.cache.normalized_values.tolist(),
            )
        ),
        columns=["Text", "Shapley Value"],
    )
)

,Text,Shapley Value
0,<|startoftext|>,0.000000
1,<|im_start|>,0.158203
2,user,0.202148
3,,0.122070
4,Who,0.089844
5,are,0.097656
6,you,0.191406
7,?,0.002258
8,<|im_end|>,0.078125
9,,0.057129


# Tests

In [11]:
explainer.shap_explainer._C

tensor([[ 0.0000,  0.0000, -0.2617,  0.0000, -0.2617,  0.3398,  0.2422,  0.0000,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.1328,  0.0586,  0.3711,  0.0000,
          0.2617,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.3398,  0.5039,  0.0000,
          0.2617,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.1094, -0.0586,  0.3945,  0.0000,
          0.2617,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.1094, -0.3398,  0.3945,  0.0000,
          0.2617,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.3945,  0.0586,  0.1094,  0.0000,
          0.2617,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.1328,  0.3398,  0.3711,  0.0000,
          0.2617,  0.0000,  0.0000],
        [ 0.0000,  0.0000, -0.2617,  0.0000,  0.0000, -0.3398,  0.5039,  0.0000,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.5039, -0

In [12]:
explainer.shap_explainer._M

tensor([[0, 0, 1, 0, 1, 2, 3, 0, 0, 0, 0],
        [0, 0, 0, 0, 2, 2, 2, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 2, 3, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 2, 3, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 2, 3, 0, 1, 0, 0],
        [0, 0, 0, 0, 2, 2, 2, 0, 1, 0, 0],
        [0, 0, 0, 0, 2, 2, 2, 0, 1, 0, 0],
        [0, 0, 1, 0, 1, 2, 3, 0, 0, 0, 0],
        [0, 0, 0, 0, 3, 2, 1, 0, 1, 0, 0],
        [0, 0, 0, 0, 2, 2, 2, 0, 1, 0, 0]], device='mps:0', dtype=torch.int16)